# 1. Import tools and paths

In [ ]:
import pandas as pd
from pathlib import Path

raw_folder = Path("raw_data")
output_folder = Path("cleaned_data")
output_folder.mkdir(exist_ok=True)

# 2. read data

In [ ]:
files = [
    raw_folder / f"{year}.csv"
    for year in range(2021, 2027)
]

data = pd.concat(
    [
        pd.read_csv(file, dtype=str, encoding="utf-8-sig")
        for file in files
    ],
    ignore_index=True
)

print("raw record", len(data))
data.head()

原始记录数： 124540


,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00013722,RALEIGH AP,2020-12-31T19:00:00-05:00,2020,12,31,19,00,35.8922,-78.7819,...,NaN,NaN,NaN,NaN,SYN09872306 11224 81303 10122 20122 30067 4022...,NaN,NaN,FM12,223,ICAO-KRDU
1,USW00013722,RALEIGH AP,2020-12-31T19:15:00-05:00,2020,12,31,19,15,35.8922,-78.7819,...,NaN,NaN,NaN,NaN,MET13612/31/20 19:15:03 SPECI KRDU 010015Z 000...,NaN,NaN,FM16,343,723060-13722
2,USW00013722,RALEIGH AP,2020-12-31T19:49:00-05:00,2020,12,31,19,49,35.8922,-78.7819,...,NaN,NaN,NaN,NaN,MET12012/31/20 19:49:03 SPECI KRDU 010049Z 350...,NaN,NaN,FM16,343,723060-13722
3,USW00013722,RALEIGH AP,2020-12-31T19:51:00-05:00,2020,12,31,19,51,35.8922,-78.7819,...,NaN,NaN,NaN,NaN,MET13712/31/20 19:51:03 METAR KRDU 010051Z 360...,NaN,NaN,FM15,343,723060-13722
4,USW00013722,RALEIGH AP,2020-12-31T20:35:00-05:00,2020,12,31,20,35,35.8922,-78.7819,...,NaN,NaN,NaN,NaN,MET11512/31/20 20:35:03 SPECI KRDU 010135Z 040...,NaN,NaN,FM16,343,723060-13722


# 3. keep needed columns and them into numbers

In [3]:
weather_columns = [
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "sea_level_pressure",
    "station_level_pressure",
    "wind_speed",
    "wind_direction"
]

data = data[["DATE"] + weather_columns].copy()

data[weather_columns] = data[weather_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# 4. keep .51

In [ ]:
# Recognize the existing -04:00 / -05:00 timezone offsets
# and convert to UTC for sorting and deduplication
data["DATE"] = pd.to_datetime(
    data["DATE"],
    format="ISO8601",
    utc=True
)

cutoff = pd.Timestamp(
    "2026-09-17 00:00:00",
    tz="America/New_York"
).tz_convert("UTC")

# Keep only data before the cutoff
data = data.loc[data["DATE"] < cutoff].copy()

# Drop rows with missing temperature values
data = data.dropna(subset=["temperature"]).copy()

# Keep only observations recorded at minute 51 of each hour
data = data.loc[data["DATE"].dt.minute == 51].copy()

# 5. Remove duplicates

In [ ]:
data = (
    data.drop_duplicates()
    .sort_values("DATE")
    .reset_index(drop=True)
)

data["hour_local"] = (
    data["DATE"]
    .dt.floor("h")
    .dt.tz_convert("America/New_York")
)

data["DATE"] = data["DATE"].dt.tz_convert(
    "America/New_York"
)

data = data[
    ["hour_local", "DATE"] + weather_columns
]

data.head()

,hour_local,DATE,temperature,dew_point_temperature,relative_humidity,sea_level_pressure,station_level_pressure,wind_speed,wind_direction
0,2020-12-31 19:00:00-05:00,2020-12-31 19:51:00-05:00,12.2,12.2,100.0,1023.1,1007.0,2.6,360.0
1,2020-12-31 20:00:00-05:00,2020-12-31 20:51:00-05:00,11.7,11.7,100.0,1023.4,1007.4,1.5,120.0
2,2020-12-31 21:00:00-05:00,2020-12-31 21:51:00-05:00,12.2,11.7,97.0,1023.6,1007.7,0.0,999.0
3,2020-12-31 22:00:00-05:00,2020-12-31 22:51:00-05:00,12.2,11.7,97.0,1023.9,1008.0,2.1,40.0
4,2020-12-31 23:00:00-05:00,2020-12-31 23:51:00-05:00,12.8,12.2,96.0,1024.0,1008.0,4.6,50.0


# 6. check

In [7]:
print("data rows after clean：", len(data))
print("start time: ", data["DATE"].min())
print("end time: ", data["DATE"].max())


print("\nlost number: ")
print(data.isna().sum())

data.to_csv(
    output_folder / "rdu_51_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n save as cleaned_data/rdu_51_clean.csv")

data rows after clean： 49881
start time:  2020-12-31 19:51:00-05:00
end time:  2026-09-16 23:51:00-04:00

lost number: 
hour_local                   0
DATE                         0
temperature                  0
dew_point_temperature        0
relative_humidity            0
sea_level_pressure         793
station_level_pressure    8421
wind_speed                  19
wind_direction             237
dtype: int64

 save as cleaned_data/rdu_51_clean.csv
